In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, ConcatDataset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.metrics import roc_auc_score

# ==========================================
# 1. SETUP & UTILITY FUNCTIONS
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_online_detector():
    """Initializes ResNet50 with unfrozen top layers for binary classification."""
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    for name, param in model.named_parameters():
        if "layer4" in name or "fc" in name:
            param.requires_grad = True
        else:
            param.requires_grad = False
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 2)
    return model

def get_transform():
    """Standard ImageNet transforms."""
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(256),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def get_split_datasets(target_path):
    """Splits a dataset folder 50/50 deterministically to prevent data leakage."""
    full_dataset = ImageFolder(root=target_path, transform=get_transform())
    train_size = int(0.5 * len(full_dataset))
    test_size = len(full_dataset) - train_size
    
    fixed_generator = torch.Generator().manual_seed(42)
    train_subset, test_subset = random_split(full_dataset, [train_size, test_size], generator=fixed_generator)
    return train_subset, test_subset

def get_eval_loader(data_dir, batch_size=32):
    """Loads the test split for GenImage, or the default test folder for CIFAKE."""
    if data_dir == '/content':
        dataset = ImageFolder(root='/content/test', transform=get_transform())
    else:
        target_path = None
        for root, dirs, files in os.walk(data_dir):
            if 'val' in dirs:
                target_path = os.path.join(root, 'val')
                break
        if not target_path:
            raise ValueError(f"Could not find 'val' in {data_dir}")
            
        _, dataset = get_split_datasets(target_path)
        
    print(f"Loading evaluation split ({len(dataset)} images) from: {data_dir}")
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

def evaluate_model(model, dataloader, phase_name="Evaluation"):
    """Evaluates the model and returns Accuracy and AuC."""
    model.eval()
    correct, total = 0, 0
    y_true, y_scores = [], []
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            probs = torch.nn.functional.softmax(outputs, dim=1)
            y_scores.extend(probs[:, 1].cpu().numpy()) 
            y_true.extend(labels.cpu().numpy())

    accuracy = 100 * correct / total
    try:
        auc = roc_auc_score(y_true, y_scores)
    except ValueError:
        auc = 0.5 
        
    print(f"[{phase_name}] Accuracy: {accuracy:.2f}% | AuC: {auc:.4f}")
    return accuracy, auc

# ==========================================
# 2. MAIN EXECUTION PIPELINE
# ==========================================
if __name__ == "__main__":
    model = get_online_detector().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

    chronological_timeline = [
        ('/content', 'CIFAKE (SD 1.4)'),
        ('/content/local_genimage/stable_diffusion_v_1_5', 'Stable Diffusion V1.5'),
        ('/content/local_genimage/Midjourney', 'Midjourney'),
        ('/content/local_genimage/glide', 'GLIDE')
    ]

    # --- RESUME CONTROLLER ---
    # Set to 0 for fresh run. Set to 1, 2, or 3 to resume from a checkpoint.
    start_step = 0 
    if start_step > 0:
        model.load_state_dict(torch.load(f'/content/detector_step_{start_step}.pth'))
        print(f"Resumed from checkpoint: Step {start_step}")
    
    cumulative_training_pool = [path for path, name in chronological_timeline[:start_step]]

    for step in range(start_step, len(chronological_timeline)):
        dataset_path, generator_name = chronological_timeline[step]
        print(f"\n{'='*50}\nSTEP {step + 1}: {generator_name}\n{'='*50}")
        
        # PHASE A: BEFORE TRAINING (Zero-Training Evaluation)
        if step > 0:
            print(f"\n[PHASE A] BEFORE TRAINING: Can it detect unseen {generator_name}?")
            try:
                eval_loader = get_eval_loader(dataset_path)
                evaluate_model(model, eval_loader, phase_name=f"Zero-Shot: {generator_name}")
            except Exception as e:
                print(f"Skipping Phase A for {generator_name}: {e}")
        else:
            print("\n[PHASE A] Base model detected. Skipping zero-training evaluation.")

        # NOTE: GLIDE is reserved strictly for Phase A validation per final paper spec.
        if generator_name == 'GLIDE':
            print("Final validation complete. Halting before training on GLIDE.")
            break

        # PHASE B: TRAINING (Update the Pool)
        print(f"\n[PHASE B] TRAINING: Adding {generator_name} to historical training pool...")
        cumulative_training_pool.append(dataset_path)
        
        epochs = 2 
        train_targets = []
        for pool_path in cumulative_training_pool:
            if pool_path == '/content':
                train_targets.append(ImageFolder(root='/content/train', transform=get_transform()))
            else:
                for root, dirs, files in os.walk(pool_path):
                    if 'val' in dirs:
                        train_subset, _ = get_split_datasets(os.path.join(root, 'val'))
                        train_targets.append(train_subset)
                        break
                        
        combined_dataset = ConcatDataset(train_targets)
        train_loader = DataLoader(combined_dataset, batch_size=32, shuffle=True, num_workers=2)

        for epoch in range(epochs):
            model.train()
            running_loss = 0.0
            for i, (inputs, labels) in enumerate(train_loader):
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
                if i % 100 == 99:
                    print(f"   Epoch [{epoch+1}/{epochs}] | Batch {i+1} | Loss: {running_loss / 100:.3f}")
                    running_loss = 0.0
                    
        # PHASE C: AFTER TRAINING (Sanity Check)
        print(f"\n[PHASE C] AFTER TRAINING: Testing accuracy on known {generator_name}")
        try:
            eval_loader = get_eval_loader(dataset_path)
            evaluate_model(model, eval_loader, phase_name=f"Post-Train: {generator_name}")
        except Exception as e:
            print(f"Skipping Phase C for {generator_name}: {e}")
                
        # PHASE D: SAVE CHECKPOINT
        checkpoint_path = f'/content/detector_step_{step+1}.pth'
        torch.save(model.state_dict(), checkpoint_path)
        print(f"\n PROGRESS SAVED: Model checkpoint secured at {checkpoint_path}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import torch

def generate_roc_curve(model, dataloader, device, evaluation_name="Evaluation"):
    """Generates and displays an ROC curve for the given model and dataloader."""
    print(f"Generating ROC Curve for: {evaluation_name}...")
    model.eval()
    y_true, y_scores = [], []
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            y_scores.extend(probs[:, 1].cpu().numpy()) 
            y_true.extend(labels.cpu().numpy())

    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.4f}')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess')
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (FPR)', fontsize=12)
    plt.ylabel('True Positive Rate (TPR)', fontsize=12)
    plt.title(f'ROC Curve: {evaluation_name}', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right", fontsize=12)
    plt.grid(alpha=0.3)
    plt.show()